In [ ]:
#!pip install gensim
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import gensim
from gensim.models import Word2Vec

In [5]:
df = pd.read_csv('/content/IMDB Dataset.csv')
df = df[:10000]
df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [6]:
def simple_preprocess(text):
    # Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Tokenization
    tokens = text.split()

    return tokens

In [7]:
tokenized_reviews = [simple_preprocess(review) for review in df['review']]

# Word2Vec Model


In [8]:
w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,   # size of word vectors
    window=5,          # context window
    min_count=2,       # ignore rare words
    workers=4
)

In [9]:
w2v_model.wv.most_similar('fantastic')

[('superb', 0.916718065738678),
 ('terrific', 0.8987188339233398),
 ('brilliant', 0.8925716876983643),
 ('outstanding', 0.8617908954620361),
 ('wonderful', 0.8575292825698853),
 ('solid', 0.8557921648025513),
 ('stunning', 0.8452707529067993),
 ('excellent', 0.8319938778877258),
 ('flawless', 0.8202100992202759),
 ('impressive', 0.8120421171188354)]

In [10]:
def get_sentence_vector(words, model):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [11]:
X = np.array([get_sentence_vector(review.split(), w2v_model) for review in df['review']])
y = df['sentiment']
y = y.map({'positive': 1, 'negative': 0})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
from sklearn.ensemble import RandomForestClassifier
lr_model = RandomForestClassifier()
lr_model.fit(X_train, y_train)

RandomForestClassifier()

In [13]:
y_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.727
